In [0]:
import csv
import os
import random
from datetime import datetime, timedelta

In [0]:
random.seed(42)
RAW_DIR = "/Workspace/Users/yashikumawat53@gmail.com/Drafts/drone_pipeline/data/raw"
os.makedirs(RAW_DIR, exist_ok=True)
MODELS = ["FalconX1", "SkyHawk200", "AeroMule", "SwiftDrone3", "CargoWing"]
CITIES = ["Downtown", "Riverside", "Hillcrest", "Northgate", "Lakeside",
          "Old Town", "Sunset Park", "East Ridge", "Harbor View", "Pine Valley"]
FAILURE_TYPES = ["FAILED_BATTERY", "FAILED_WEATHER", "FAILED_SIGNAL"]

In [0]:
# ---------- 1. drones.csv ----------
N_DRONES = 25
drones = []
for drone_id in range(1, N_DRONES + 1):
    drones.append({
        "drone_id": drone_id,
        "model": random.choice(MODELS),
        "max_range_km": round(random.uniform(15, 60), 1),
    })

with open(f"{RAW_DIR}/drones.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["drone_id", "model", "max_range_km"])
    writer.writeheader()
    writer.writerows(drones)

In [0]:
# ---------- 2. deliveries.csv ----------
N_DELIVERIES = 600
base_time = datetime(2026, 6, 1, 8, 0, 0)
deliveries = []
for delivery_id in range(1, N_DELIVERIES + 1):
    drone_id = random.randint(1, N_DRONES)
    source, destination = random.sample(CITIES, 2)
    distance_km = round(random.uniform(2, 25), 2)
    start_time = base_time + timedelta(minutes=random.randint(0, 60 * 24 * 20))
    duration_min = distance_km * random.uniform(2.0, 4.5) + random.uniform(-3, 8)
    end_time = start_time + timedelta(minutes=max(duration_min, 1))
    battery_consumed = round(distance_km * random.uniform(1.5, 3.0), 2)
    deliveries.append({
        "delivery_id": delivery_id,
        "drone_id": drone_id,
        "source": source,
        "destination": destination,
        "distance_km": distance_km,
        "start_time": start_time.strftime("%Y-%m-%d %H:%M:%S"),
        "end_time": end_time.strftime("%Y-%m-%d %H:%M:%S"),
        "battery_consumed": battery_consumed,
    })

with open(f"{RAW_DIR}/deliveries.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(deliveries[0].keys()))
    writer.writeheader()
    writer.writerows(deliveries)


In [0]:
# ---------- 3. flight_logs.csv ----------

flight_logs = []
log_id = 1
for d in deliveries:
    battery_level = round(max(5, 100 - d["battery_consumed"] - random.uniform(0, 10)), 1)
    gps_signal = round(random.uniform(20, 100), 1)

    roll = random.random()
    if battery_level < 15 and roll < 0.6:
        status = "FAILED_BATTERY"
    elif gps_signal < 35 and roll < 0.5:
        status = "FAILED_SIGNAL"
    elif roll < 0.06:
        status = "FAILED_WEATHER"
    else:
        status = "SUCCESS"

    flight_logs.append({
        "log_id": log_id,
        "delivery_id": d["delivery_id"],
        "drone_id": d["drone_id"],
        "battery_level": battery_level,
        "gps_signal": gps_signal,
        "status": status,
    })
    log_id += 1

with open(f"{RAW_DIR}/flight_logs.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(flight_logs[0].keys()))
    writer.writeheader()
    writer.writerows(flight_logs)

In [0]:
print(f"Generated {N_DRONES} drones, {N_DELIVERIES} deliveries, {len(flight_logs)} flight logs in {RAW_DIR}")

Generated 25 drones, 600 deliveries, 600 flight logs in /Workspace/Users/yashikumawat53@gmail.com/Drafts/drone_pipeline/data/raw
